# 01 · Farmer Intelligence: EDA and Preprocessing

**Dataset:** Crop-wise Area, Production and Yield (`data/raw/farmer/crop-wise-area-production-yield.csv`)

**Output:** `data/cleaned/farmer/crop_production_cleaned.csv`, read by `src/models/train_farmer.py` and the Farmer Intelligence page.

**Rules followed:** the raw file is read-only and is never overwritten. Unusual but valid rows are kept. Cleaning decisions are written in markdown next to the code.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Works whether Jupyter is opened from the project root or from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)

Project root: /Users/kumarsidharth/AgriLink 7


## 1. Load the raw dataset

In [2]:
RAW_PATH = ROOT / "data" / "raw" / "farmer" / "crop-wise-area-production-yield.csv"
CLEAN_PATH = ROOT / "data" / "cleaned" / "farmer" / "crop_production_cleaned.csv"

df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print("Shape (rows, columns):", df_raw.shape)
print("File size (MB):", round(RAW_PATH.stat().st_size / 1e6, 1))
print("Columns:", list(df_raw.columns))
display(df_raw.head())

Shape (rows, columns): (455359, 16)
File size (MB): 56.2
Columns: ['id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit']


,id,year,state_name,state_code,district_name,district_code,crop_name,crop_code,crop_type,season,area,area_unit,production,production_unit,yield,yield_unit
0,0,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Arhar/Tur,202.0,Pulses,Kharif,21400.0,Hectare,2600.0,Tonnes,0.121,Tonnes/Hectare
1,1,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Bajra,103.0,Cereals,Kharif,1400.0,Hectare,500.0,Tonnes,0.357,Tonnes/Hectare
2,2,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Castor Seed,1002.0,Oilseeds,Kharif,1000.0,Hectare,100.0,Tonnes,0.100,Tonnes/Hectare
3,3,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Cotton(Lint),1101.0,Fiber Crops,Kharif,7300.0,Hectare,9400.0,Tonnes,1.288,Tonnes/Hectare
4,4,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Dry Chillies,502.0,Spices,Kharif,3700.0,Hectare,7100.0,Tonnes,1.919,Tonnes/Hectare


## 2. Structural inspection: types, missing values, duplicates, unique values

In [3]:
df_raw.info()
print("\nMissing values per column:")
print(df_raw.isna().sum())
print("\nExact duplicate rows:", df_raw.duplicated().sum())
print("\nDistinct values:")
for col in ["year", "state_name", "district_name", "crop_name", "season"]:
    print(f"  {col}: {df_raw[col].nunique()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 455359 entries, 0 to 455358
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               455359 non-null  int64  
 1   year             455359 non-null  object 
 2   state_name       455359 non-null  object 
 3   state_code       455359 non-null  int64  
 4   district_name    455359 non-null  object 
 5   district_code    455359 non-null  int64  
 6   crop_name        455359 non-null  object 
 7   crop_code        455338 non-null  float64
 8   crop_type        455359 non-null  object 
 9   season           455359 non-null  object 
 10  area             455359 non-null  float64
 11  area_unit        455359 non-null  object 
 12  production       450350 non-null  float64
 13  production_unit  455359 non-null  object 
 14  yield            455359 non-null  float64
 15  yield_unit       455359 non-null  object 
dtypes: float64(4), int64(3), object(9)
mem

## 3. Year coverage

Agricultural years look like `2001-2002`. The models and the app use the **starting year** as a number, so the year range is checked before any filtering.

In [ ]:
year_start = pd.to_numeric(df_raw["year"].astype(str).str[:4], errors="coerce")
print("Unparsable year values:", year_start.isna().sum())
print("Earliest starting year:", year_start.min(), "| latest:", year_start.max())
print("\nRows per starting year:")
print(year_start.value_counts().sort_index().to_string())

## 4. Numerical validation

Negative or zero values in `area`, `production` and `yield` can indicate data-entry problems. Zero production is kept because it can be a real outcome. Only clearly impossible values would be removed.

In [ ]:
numeric_cols = ["area", "production", "yield"]
display(df_raw[numeric_cols].describe().T)
print("Negative area:", (df_raw["area"] < 0).sum())
print("Negative production:", (df_raw["production"] < 0).sum())
print("Zero production rows:", (df_raw["production"] == 0).sum())
print("'Total' season rows (kept, excluded at modelling time):", (df_raw["season"].str.strip().str.casefold() == "total").sum())

## 5. Cleaning decisions

1. **Keep agricultural years 2001-02 to 2022-23.** This matches the range used by the trained model and the app. Earlier years are historical records and are not needed for the current model.
2. **Strip whitespace** from text columns so that `"Assam "` and `"Assam"` are not treated as different states.
3. **Convert numeric columns** with `errors="coerce"`. Anything that cannot be read as a number becomes missing, and it is not guessed.
4. **Remove exact duplicate rows only.** Repeated identical records would double-count production.
5. **Keep `Total` rows.** They are excluded in the training code so they are not double-counted.

In [ ]:
df = df_raw.copy()

# 1. Agricultural-year range used by the model and the app
start = pd.to_numeric(df["year"].astype(str).str[:4], errors="coerce")
df = df[(start >= 2001) & (start <= 2022)].copy()

# 2. Standardise text columns
for col in ["state_name", "district_name", "crop_name", "season", "year"]:
    df[col] = df[col].astype("string").str.strip()

# 3. Numeric conversion
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Exact duplicates
before = len(df)
df = df.drop_duplicates()
print("Duplicate rows removed:", before - len(df))
print("Rows after cleaning:", len(df), "| columns:", df.shape[1])

## 6. Check the cleaned data and save it

In [ ]:
print("Missing values after cleaning:")
print(df[numeric_cols].isna().sum())
print("\nYears covered:", df["year"].min(), "to", df["year"].max())
print("States:", df["state_name"].nunique(), "| Districts:", df["district_name"].nunique(), "| Crops:", df["crop_name"].nunique())

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("Saved cleaned dataset:", CLEAN_PATH)

## 7. Quick EDA: top crops by total production

In [ ]:
clean_for_eda = df[df["season"].str.casefold() != "total"]
top = (clean_for_eda.groupby("crop_name")["production"].sum()
       .sort_values(ascending=False).head(10))
display(top.to_frame("total_production_tonnes").round(0))

fig, ax = plt.subplots(figsize=(9, 5))
top.sort_values().plot(kind="barh", ax=ax, color="#3B8F7B")
ax.set_title("Top 10 crops by total production (2001-2022, excluding Total rows)")
ax.set_xlabel("Production (tonnes)")
plt.tight_layout()
plt.show()